# AC-OPF Objective Check: Pyomo/IPOPT vs PYPOWER on IEEE Cases

Does the IPOPT solve of the Pyomo AC-OPF formulation from [03_acopf_pyomo.ipynb](https://github.com/vjkhtw/pjm5-opf/blob/main/notebooks/03_acopf_pyomo.ipynb) reach the same objective as PYPOWER's interior-point solver ([04_acopf_pypower_reference.ipynb](https://github.com/vjkhtw/pjm5-opf/blob/main/notebooks/04_acopf_pypower_reference.ipynb))? Only the **total generation cost** and how long does each take? Only the **total generation cost** and the **solve time** are compared, on six IEEE cases from PGLib-OPF (typical operations): 14, 30, 39 (EPRI), 57, 118 and 162 (DTC) buses.

**Contents**
* Setup
* Data pipeline: case file to solver input
* Network features beyond the 5-bus case: tap ratios and bus shunts
* Pyomo AC-OPF model
* Sanity check on the 5-bus case
* Objective comparison across IEEE cases
* Solve

## Setup

In [1]:
!idaes get-extensions


Getting files...
Done
-----------------------------------------------------------------
IDAES Extensions Build Versions
Solvers:  v3.4.2 20240811 ubuntu2204-x86_64
Library:  v3.4.2 20240811 ubuntu2204-x86_64



In [2]:
import os, re, time, urllib.request
import idaes
import numpy as np, pandas as pd
import pyomo.environ as pyo
from pypower.api import runopf, ppoption

if idaes.bin_directory not in os.environ.get("PATH", ""):
    os.environ["PATH"] = idaes.bin_directory + os.pathsep + os.environ.get("PATH", "")

## Data pipeline: case file to solver input
Both solvers are fed from the same Python dictionary, so they always see identical data:

`pglib-opf .m file` → `read_matpower()` → `case` dict → `solve_pyomo_acopf(case)` and `solve_pypower(case)`

The dictionary holds `baseMVA` and four NumPy arrays in MATPOWER column order: `bus`, `gen`, `gencost`, `branch`. PYPOWER reads this dictionary directly. The Pyomo function reads the same arrays. To use your own network, build a dictionary with these keys and column layouts (see the README).

In [3]:
BASE_URL = "https://raw.githubusercontent.com/power-grid-lib/pglib-opf/master/"

def read_matpower(name):
    """Download a pglib-opf case (e.g. 'pglib_opf_case14_ieee.m') and return the MATPOWER-style dict."""
    txt = urllib.request.urlopen(BASE_URL + name).read().decode()
    txt = "\n".join(l.split("%")[0] for l in txt.splitlines())          # strip comments
    case = {"baseMVA": float(re.search(r"mpc\.baseMVA\s*=\s*([\d.]+)", txt).group(1))}
    for key in ["bus", "gen", "gencost", "branch"]:
        body = re.search(r"mpc\.%s\s*=\s*\[(.*?)\];" % key, txt, re.S).group(1)
        case[key] = np.array([r.split() for r in body.replace("\t", " ").split(";") if r.strip()], dtype=float)
    return case

CASES = ["14_ieee", "30_ieee", "39_epri", "57_ieee", "118_ieee", "162_ieee_dtc"]
cases = {n: read_matpower(f"pglib_opf_case{n}.m") for n in CASES}

## Network features beyond the 5-bus case: tap ratios and bus shunts
The 5-bus case has only plain lines. The IEEE cases also contain two features that the power-balance and flow equations must include, otherwise Pyomo would be solving a different network from PYPOWER. Neither is a new optimisation constraint.

**Transformer tap ratio $\tau$.** A transformer changes the voltage level, so one side of the branch sees the voltage scaled by $\tau$. The file stores the ratio in the `branch` array, where 0 means no tap ($\tau = 1$). With series admittance $y = g + jb$ and line charging $b_c$, the branch admittances become

$$Y_{ff} = \frac{y + j b_c/2}{\tau^2}, \qquad Y_{ft} = Y_{tf} = -\frac{y}{\tau}, \qquad Y_{tt} = y + j\,\frac{b_c}{2}$$

For $\tau = 1$ these reduce to the ordinary line equations used in the 5-bus notebook.

**Bus shunts $G_s, B_s$.** Fixed admittances from a bus to ground, such as capacitor banks for voltage support. The file gives them in MW and MVAr at 1.0 p.u. voltage. The power they draw scales with $V_i^2$, so they enter as an extra load at the bus:
* active balance: load term $G_s V_i^2$
* reactive balance: load term $-B_s V_i^2$ (a positive $B_s$ is a capacitor and injects reactive power)

In [4]:
rows = []
for n, c in cases.items():
    br, bus = c["branch"], c["bus"]
    rows.append({"Case": n, "Buses": len(bus), "Generators": len(c["gen"]), "Branches": len(br),
                 "Tapped branches": int((br[:, 8] != 0).sum()),
                 "Buses with shunts": int(((bus[:, 4] != 0) | (bus[:, 5] != 0)).sum()),
                 "Demand (MW)": bus[:, 2].sum()})
pd.DataFrame(rows).style.format({"Demand (MW)": "{:,.0f}"}).hide(axis="index")

Case,Buses,Generators,Branches,Tapped branches,Buses with shunts,Demand (MW)
14_ieee,14,5,20,3,1,259
30_ieee,30,6,41,7,2,283
39_epri,39,10,46,11,0,"6,254"
57_ieee,57,7,80,17,3,"1,251"
118_ieee,118,54,186,11,14,"4,242"
162_ieee_dtc,162,12,284,91,34,"7,239"


## Pyomo AC-OPF model
The formulation from [03_acopf_pyomo.ipynb](https://github.com/vjkhtw/pjm5-opf/blob/main/notebooks/03_acopf_pyomo.ipynb), extended with the tap ratio and bus shunt terms above: minimise $\sum_g c_g P_g$ subject to active and reactive nodal balance, generator P/Q limits, voltage limits, apparent-power limits at both line ends, branch angle limits and a reference angle. The start is flat: $V=1$, $\theta=0$, $P_g$ at the midpoint of its limits, $Q_g$ at 0 clipped into its limits (some generators have $Q_{min} > 0$, and Pyomo warns about initial values outside the bounds). The function returns only the solver status and the total cost.

In [5]:
def solve_pyomo_acopf(case, tol=1e-8):
    base = case["baseMVA"]; bus, gen, gcost, br = case["bus"], case["gen"], case["gencost"], case["branch"]
    nb, ng, nl = len(bus), len(gen), len(br)
    idx = {int(b): k for k, b in enumerate(bus[:, 0])}
    ref = idx[int(bus[bus[:, 1] == 3, 0][0])]
    assert np.allclose(gcost[:, 4], 0), "only linear costs (c2 = 0) are handled"
    assert np.allclose(br[:, 9], 0), "phase-shifting transformers are not handled"
    assert np.allclose(gcost[:, 6], 0), "constant cost term c0 is not in the objective"
    assert (gen[:, 7] == 1).all(), "out-of-service generators (status = 0) are not handled"
    assert (br[:, 10] == 1).all(), "out-of-service branches (status = 0) are not handled"

    Pd, Qd, Gs, Bs = bus[:, 2] / base, bus[:, 3] / base, bus[:, 4] / base, bus[:, 5] / base
    Vmax, Vmin = bus[:, 11], bus[:, 12]
    gbus = [idx[int(b)] for b in gen[:, 0]]
    Pmax, Pmin, Qmax, Qmin = gen[:, 8] / base, gen[:, 9] / base, gen[:, 3] / base, gen[:, 4] / base
    c1 = gcost[:, 5]
    f = [idx[int(b)] for b in br[:, 0]]; t = [idx[int(b)] for b in br[:, 1]]
    tap = np.where(br[:, 8] == 0, 1.0, br[:, 8]); rate = br[:, 5] / base
    den = br[:, 2]**2 + br[:, 3]**2; g, b, bc = br[:, 2] / den, -br[:, 3] / den, br[:, 4]
    Gff, Bff, Gft, Bft, Gtt, Btt = g / tap**2, (b + bc / 2) / tap**2, -g / tap, -b / tap, g, b + bc / 2

    m = pyo.ConcreteModel()
    m.B, m.G, m.L = pyo.RangeSet(0, nb - 1), pyo.RangeSet(0, ng - 1), pyo.RangeSet(0, nl - 1)
    m.rate = pyo.Param(m.L, mutable=True, initialize=dict(enumerate(rate)))
    m.Pg = pyo.Var(m.G, bounds=lambda m, k: (Pmin[k], Pmax[k]), initialize=lambda m, k: (Pmin[k] + Pmax[k]) / 2)
    m.Qg = pyo.Var(m.G, bounds=lambda m, k: (Qmin[k], Qmax[k]), initialize=lambda m, k: float(np.clip(0.0, Qmin[k], Qmax[k])))
    m.V = pyo.Var(m.B, bounds=lambda m, i: (Vmin[i], Vmax[i]), initialize=1.0)
    m.th = pyo.Var(m.B, bounds=(-np.pi, np.pi), initialize=0.0)
    m.obj = pyo.Objective(expr=base * sum(c1[k] * m.Pg[k] for k in m.G))

    def Pf(m, l): i, j = f[l], t[l]; d = m.th[i] - m.th[j]; return Gff[l]*m.V[i]**2 + m.V[i]*m.V[j]*(Gft[l]*pyo.cos(d) + Bft[l]*pyo.sin(d))
    def Qf(m, l): i, j = f[l], t[l]; d = m.th[i] - m.th[j]; return -Bff[l]*m.V[i]**2 + m.V[i]*m.V[j]*(Gft[l]*pyo.sin(d) - Bft[l]*pyo.cos(d))
    def Pt(m, l): i, j = f[l], t[l]; d = m.th[j] - m.th[i]; return Gtt[l]*m.V[j]**2 + m.V[i]*m.V[j]*(Gft[l]*pyo.cos(d) + Bft[l]*pyo.sin(d))
    def Qt(m, l): i, j = f[l], t[l]; d = m.th[j] - m.th[i]; return -Btt[l]*m.V[j]**2 + m.V[i]*m.V[j]*(Gft[l]*pyo.sin(d) - Bft[l]*pyo.cos(d))

    gens_at = {i: [k for k in range(ng) if gbus[k] == i] for i in range(nb)}
    from_at = {i: [l for l in range(nl) if f[l] == i] for i in range(nb)}
    to_at   = {i: [l for l in range(nl) if t[l] == i] for i in range(nb)}
    m.balP = pyo.Constraint(m.B, rule=lambda m, i: sum(m.Pg[k] for k in gens_at[i]) - Pd[i] - Gs[i]*m.V[i]**2
                            - sum(Pf(m, l) for l in from_at[i]) - sum(Pt(m, l) for l in to_at[i]) == 0)
    m.balQ = pyo.Constraint(m.B, rule=lambda m, i: sum(m.Qg[k] for k in gens_at[i]) - Qd[i] + Bs[i]*m.V[i]**2
                            - sum(Qf(m, l) for l in from_at[i]) - sum(Qt(m, l) for l in to_at[i]) == 0)
    m.lim_f = pyo.Constraint(m.L, rule=lambda m, l: Pf(m, l)**2 + Qf(m, l)**2 <= m.rate[l]**2)
    m.lim_t = pyo.Constraint(m.L, rule=lambda m, l: Pt(m, l)**2 + Qt(m, l)**2 <= m.rate[l]**2)
    m.ang = pyo.Constraint(m.L, rule=lambda m, l: (np.radians(br[l, 11]), m.th[f[l]] - m.th[t[l]], np.radians(br[l, 12])))
    m.refang = pyo.Constraint(expr=m.th[ref] == 0)

    solver = pyo.SolverFactory("ipopt"); solver.options["tol"] = tol; solver.options["max_iter"] = 3000
    res = solver.solve(m)
    return {"status": str(res.solver.termination_condition), "cost": pyo.value(m.obj)}


def solve_pypower(case):
    c = {k: (v.copy() if hasattr(v, "copy") else v) for k, v in case.items()}; c["version"] = "2"
    r = runopf(c, ppoption(VERBOSE=0, OUT_ALL=0))
    return {"success": bool(r["success"]), "cost": float(r["f"])}

## Sanity check on the 5-bus case
The same function on the PJM 5-bus data (no taps, no shunts) should return the objective reported in `03_acopf_pyomo.ipynb`, \$17,551.89/h.

In [6]:
pjm5 = {"baseMVA": 100.0,
  "bus": np.array([[1,2,0,0,0,0,1,1,0,230,1,1.1,.9],[2,1,300,98.61,0,0,1,1,0,230,1,1.1,.9],[3,2,300,98.61,0,0,1,1,0,230,1,1.1,.9],
                   [4,3,400,131.47,0,0,1,1,0,230,1,1.1,.9],[5,2,0,0,0,0,1,1,0,230,1,1.1,.9]], float),
  "gen": np.array([[1,20,0,30,-30,1,100,1,40,0],[1,85,0,127.5,-127.5,1,100,1,170,0],[3,260,0,390,-390,1,100,1,520,0],
                   [4,100,0,150,-150,1,100,1,200,0],[5,300,0,450,-450,1,100,1,600,0]], float),
  "gencost": np.array([[2,0,0,3,0,c,0] for c in (14,15,30,40,10)], float),
  "branch": np.array([[1,2,.00281,.0281,.00712,400,400,400,0,0,1,-30,30],[1,4,.00304,.0304,.00658,426,426,426,0,0,1,-30,30],
                      [1,5,.00064,.0064,.03126,426,426,426,0,0,1,-30,30],[2,3,.00108,.0108,.01852,426,426,426,0,0,1,-30,30],
                      [3,4,.00297,.0297,.00674,426,426,426,0,0,1,-30,30],[4,5,.00297,.0297,.00674,240,240,240,0,0,1,-30,30]], float)}
o5, p5 = solve_pyomo_acopf(pjm5), solve_pypower(pjm5)
print(f"5-bus  Pyomo/IPOPT ${o5['cost']:,.4f}/h | PYPOWER ${p5['cost']:,.4f}/h | notebook 03: $17,551.89/h")

5-bus  Pyomo/IPOPT $17,551.8908/h | PYPOWER $17,551.8915/h | notebook 03: $17,551.89/h


## Objective comparison across IEEE cases
Each case is solved with both solvers. IPOPT starts from the flat start described above and PYPOWER from the setpoints in the case file, so the two do not share a starting point.

In [7]:
rows = []
for n, case in cases.items():
    t0 = time.time(); o = solve_pyomo_acopf(case); t_o = time.time() - t0
    t0 = time.time(); p = solve_pypower(case);     t_p = time.time() - t0
    rows.append({"Case": n, "Buses": len(case["bus"]),
                 "IPOPT status": o["status"], "PYPOWER converged": p["success"],
                 "Pyomo/IPOPT cost ($/h)": o["cost"], "PYPOWER cost ($/h)": p["cost"],
                 "Abs. diff ($/h)": abs(o["cost"] - p["cost"]), "Rel. diff": abs(o["cost"] - p["cost"]) / p["cost"],
                 "IPOPT time (s)": t_o, "PYPOWER time (s)": t_p})
results = pd.DataFrame(rows)
results.style.format({"Pyomo/IPOPT cost ($/h)": "{:,.4f}", "PYPOWER cost ($/h)": "{:,.4f}", "Abs. diff ($/h)": "{:.1e}",
                      "Rel. diff": "{:.1e}", "IPOPT time (s)": "{:.2f}", "PYPOWER time (s)": "{:.2f}"}).hide(axis="index")

Case,Buses,IPOPT status,PYPOWER converged,Pyomo/IPOPT cost ($/h),PYPOWER cost ($/h),Abs. diff ($/h),Rel. diff,IPOPT time (s),PYPOWER time (s)
14_ieee,14,optimal,True,"2,178.0804","2,178.0805",1.1e-04,5.2e-08,0.07,0.52
30_ieee,30,optimal,True,"8,208.5154","8,208.5152",2.7e-04,3.3e-08,0.10,0.47
39_epri,39,optimal,True,"138,415.5613","138,415.5633",2.0e-03,1.4e-08,0.15,0.81
57_ieee,57,optimal,True,"37,589.3381","37,589.3390",8.5e-04,2.3e-08,0.17,0.60
118_ieee,118,optimal,True,"97,213.6065","97,213.6079",1.4e-03,1.4e-08,0.41,1.20
162_ieee_dtc,162,optimal,True,"108,075.6435","108,075.6482",4.7e-03,4.4e-08,0.73,1.33


In [8]:
print(f"Largest relative difference over {len(results)} cases: {results['Rel. diff'].max():.1e}")
print(f"Largest absolute difference: ${results['Abs. diff ($/h)'].max():.4f}/h")
print("All converged:", bool((results['IPOPT status'] == 'optimal').all() and results['PYPOWER converged'].all()))

Largest relative difference over 6 cases: 5.2e-08
Largest absolute difference: $0.0047/h
All converged: True


## Solve time
The table above shows a single run per case, which is noisy. Here each case is run once as a warm-up (not counted) and then `N_RUNS` times; the table shows the **median** in seconds. Times are wall-clock on the machine running the notebook, so absolute values will differ on yours.

* **Pyomo/IPOPT total**: building the Pyomo model plus the IPOPT solve (including writing the model for IPOPT and reading the result back).
* **PYPOWER total**: the whole `runopf` call, including its own case conversion and set-up.

These compare two complete workflows, not the bare solvers.

In [9]:
N_RUNS = 5
trows = []
for n, case in cases.items():
    solve_pyomo_acopf(case); solve_pypower(case)                          # warm-up
    t_o, t_p = [], []
    for _ in range(N_RUNS):
        t0 = time.perf_counter(); solve_pyomo_acopf(case); t_o.append(time.perf_counter() - t0)
        t0 = time.perf_counter(); solve_pypower(case);     t_p.append(time.perf_counter() - t0)
    trows.append({"Case": n, "Buses": len(case["bus"]), "Pyomo/IPOPT total (s)": np.median(t_o), "PYPOWER total (s)": np.median(t_p)})
timing = pd.DataFrame(trows)
print(f"Median of {N_RUNS} runs per case after one warm-up run.")
timing.style.format({"Pyomo/IPOPT total (s)": "{:.3f}", "PYPOWER total (s)": "{:.3f}"}).hide(axis="index")

Median of 5 runs per case after one warm-up run.


Case,Buses,Pyomo/IPOPT total (s),PYPOWER total (s)
14_ieee,14,0.064,0.529
30_ieee,30,0.103,0.472
39_epri,39,0.161,0.821
57_ieee,57,0.170,0.594
118_ieee,118,0.421,1.091
162_ieee_dtc,162,0.722,1.346


**Reading the result.** The objectives agree to at most 5.2e-8 relative on every case (largest absolute difference \$0.0047/h). The remaining gap is likely consistent with PYPOWER's default convergence tolerances (around 1e-6), but that was not tested. Agreement means both solvers reach the same local optimum from different starting points; AC-OPF is non-convex, so it does not prove a global optimum.

Cases not included: IEEE 24 and 73 (quadratic generator costs) and IEEE 300 (phase-shifting transformers). Using them would require changing the objective or the branch model. The model also asserts that the constant cost term $c_0$ is zero and that all generators and branches are in service.